# Signal-Space Image Generation: V6

---

## Last run on Kaggle (read first)

**Before you run:** enable **Internet** on the notebook settings; add Hugging Face secret **`HF_TOKEN`** (same as V5).

**Suggested order**

1. Run cells **1 → 14** (imports through dataset + CLIP). One-time setup (~15–30 min including HF cache).
2. **Section 14** sanity check (param count + aux disabled).
3. **Section 15** — 30-epoch `no_seq_pe` baseline (~60–90 min T4). Saves under `checkpoints_v6_baseline_30ep/`. Uses `resume_from` only if that folder already has `latest.pt` (fresh run: folder empty).
4. **Section 16** — postulate grid (~**6–10 h** T4; **P3 beam perturb** can be **2–3× slower** per epoch because of batched `einsum` in the renderer). Each seed saves under `checkpoints_v6_ablation/<run_name>/seed_<s>/`.
5. **Section 17** — `decide(results)` (instant).

**Can you run everything in one Kaggle session?**

- **Often risky.** Section 15 + Section 16 together are typically **~7–12+ hours** on a T4; free-tier limits vary and **P3** stretches the upper bound.
- **Safer:** Session **A** = setup + sanity + Section **15** (optional if you already trust V5 numbers — skip and run **16** only). Session **B** = Section **16** + **17** only.
- If the session dies mid-grid, each run still has `latest.pt` under its own folder; re-run Section **16** after trimming the grid or lowering epochs.

**Likelihood vs postulates (important):** Training optimizes **DMoL NLL on scan signals** (same as V5). Auxiliary renderer heads (**P1/P2**) do **not** change that objective — metric differences vs baseline come mainly from **extra parameters / optimisation dynamics**. **P3** adds an **L2 penalty** on `pos_deltas`, so it can shift training.

---


## What V6 adds over V5

V5 produced a multi-seed-verified finding: removing the 1D sequence positional
encoding improves bpd_pixel by ~0.67 (44 sigma significance). V6 builds on that
result by testing three additional CRT-inspired architectural choices proposed
by the project's co-author, each as a Config toggle that can be ablated.

**Postulate 1 (`use_learned_sigma`).** The CRT renderer's beam width is fixed
in V5. In a real CRT it varies dynamically with electron-gun focus. We extend
the DMoL head to also predict a per-step sigma, and wire the renderer to use
the predicted sigma when generating images.

**Postulate 2 (`use_persistence`).** Real phosphors decay with finite
persistence rather than discrete per-step deposition. We add a causal
smoothing step inside the renderer with a learnable persistence parameter.

**Postulate 3 (`use_beam_perturb`).** V5 ties the model to a fixed raster
path. In a real CRT, the deflection signal can place the beam off-grid. We
let the model predict small (x, y) deltas per step. Note: this directly
challenges the V5 path-PE finding -- if the path is not deterministic, the
path PE may carry less information.

## Methodology

Every new toggle defaults to False. With all toggles off, V6 == V5 numerically.
Each toggle is tested independently with 3 seeds at 10 epochs against the
no_seq_pe baseline. A toggle is adopted only if it improves bpd_pixel by more
than 3 sigma. Toggles that fail are still reported in the paper as negative
results.

## What V6 is *not*

A pivot to VAE objective. Co-author's parallel notebooks (Conv1D-VAE,
physics-VAE) explore the VAE direction; those produce qualitative samples but
cannot be compared against PixelCNN++ on bpd. V6 stays AR + DMoL because that
is the only path to a likelihood-based comparison with prior work.

## 1. Environment setup

In [1]:
!pip install -q ftfy regex tqdm
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import os

if os.path.isdir("/kaggle/working"):
    WORK_DIR = "/kaggle/working"
elif os.path.isdir("/content"):
    WORK_DIR = "/content"
else:
    WORK_DIR = os.getcwd()

hf_token = os.environ.get("HF_TOKEN")
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN") or hf_token
except Exception:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN") or hf_token
    except Exception:
        pass

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print(f"HF_TOKEN set, WORK_DIR={WORK_DIR}")
else:
    print(f"HF_TOKEN not set; HF datasets may fail. WORK_DIR={WORK_DIR}")


HF_TOKEN set, WORK_DIR=/kaggle/working


In [3]:
import math
import os
from dataclasses import dataclass, field
from typing import Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

## 2. Configuration

V6 adds three new flags at the bottom of the Config:

- `use_learned_sigma` (P1): adds 1 output per step to the DMoL head for
  per-step sigma prediction; renderer uses predicted sigma when set.
- `use_persistence` (P2): renderer applies causal smoothing along the
  scan with a learnable decay.
- `use_beam_perturb` (P3): adds 2 outputs per step (dx, dy) for beam
  position offsets; renderer reads predicted positions instead of the
  fixed raster path positions.

All three default to False so V6 matches V5 exactly when run as-is.

In [4]:
@dataclass
class Config:
    # Image and scan geometry
    image_size: int = 32
    channels: int = 3
    samples_per_pixel: int = 2
    flyback_frac: float = 0.08

    # Renderer (V5)
    beam_sigma: float = 0.75

    # Conditioning
    clip_dim: int = 512
    cond_dim: int = 256

    # Transformer
    d_model: int = 256
    n_heads: int = 4
    n_layers: int = 6
    ff_mult: int = 4
    dropout: float = 0.1

    # Output head (discretized mixture of logistics)
    n_mixtures: int = 5

    # Training
    batch_size: int = 24
    epochs: int = 30
    lr: float = 3e-4
    weight_decay: float = 1e-2
    grad_clip: float = 1.0
    warmup_steps: int = 500
    ss_max: float = 0.25

    # V5 ablation toggles (PE / FiLM / flyback)
    use_path_pos_enc: bool = True
    use_seq_pos_enc: bool = True   # Set False for V5 best model
    use_flyback_mask: bool = True
    use_film: bool = True

    # ---- V6 NEW: postulate toggles (all default False -> V6 == V5) ----
    # P1: per-step learned beam focus
    use_learned_sigma: bool = False
    sigma_min: float = 0.3
    sigma_max: float = 1.5

    # P2: phosphor persistence (causal smoothing on rendered output)
    use_persistence: bool = False
    persistence_init: float = 0.1   # initial decay; 0 = no persistence

    # P3: beam position perturbation (delta x, delta y per step)
    use_beam_perturb: bool = False
    beam_perturb_max: float = 0.5   # max offset in pixels

    seed: int = 0
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


cfg = Config()
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
print(f"device: {cfg.device}")
print(f"V6 postulates: learned_sigma={cfg.use_learned_sigma}, "
      f"persistence={cfg.use_persistence}, beam_perturb={cfg.use_beam_perturb}")

device: cuda
V6 postulates: learned_sigma=False, persistence=False, beam_perturb=False


## 3. Scan path

Identical to V5: row-major raster scan with sub-pixel sampling and flyback
gaps marked in `beam_on`.

In [5]:
def generate_raster_path(w, h, samples_per_pixel, flyback_frac):
    samples_per_row = w * samples_per_pixel
    flyback_samples = max(int(samples_per_row * flyback_frac), 1)

    xs, ys, on = [], [], []
    for row in range(h):
        y = row + 0.5
        for i in range(samples_per_row):
            x = (i / max(samples_per_row - 1, 1)) * (w - 1)
            xs.append(x); ys.append(y); on.append(True)
        for i in range(flyback_samples):
            t = (i + 1) / flyback_samples
            x = (1 - t) * (w - 1)
            xs.append(x); ys.append(y); on.append(False)

    path = np.stack([np.array(xs, np.float32), np.array(ys, np.float32)], axis=1)
    beam_on = np.array(on, dtype=bool)
    return path, beam_on


path_np, beam_on_np = generate_raster_path(
    cfg.image_size, cfg.image_size, cfg.samples_per_pixel, cfg.flyback_frac
)
SEQ_LEN = len(path_np)
print(f"sequence length: {SEQ_LEN}")
print(f"active fraction: {beam_on_np.mean():.3f}")

sequence length: 2208
active fraction: 0.928


## 4. CRT renderer (extended)

The V6 renderer accepts three optional inputs corresponding to the postulates:

- `sigma_per_step` (P1): if provided, replaces `self.sigma` per position.
- `pos_deltas` (P2): if provided, shifts the path positions.
- `persistence_decay` (parameter, used if `use_persistence`): causal smoothing
  applied to the rendered image along the scan order.

When all three are off, the rendering is bit-identical to V5 (verified in the
sanity check below).

In [6]:
class CRTRenderer(nn.Module):
    def __init__(self, path: np.ndarray, beam_on: np.ndarray,
                 image_size: int, sigma: float, cfg: Optional[Config] = None):
        super().__init__()
        self.image_size = image_size
        self.sigma = sigma
        self.use_persistence = bool(cfg.use_persistence) if cfg is not None else False

        self.register_buffer("path_x", torch.from_numpy(path[:, 0]).float())
        self.register_buffer("path_y", torch.from_numpy(path[:, 1]).float())
        self.register_buffer(
            "beam_on_f", torch.from_numpy(beam_on.astype(np.float32))
        )

        ys, xs = torch.meshgrid(
            torch.arange(image_size, dtype=torch.float32),
            torch.arange(image_size, dtype=torch.float32),
            indexing="ij",
        )
        self.register_buffer("grid_x", xs)
        self.register_buffer("grid_y", ys)

        # Fixed gain for V5 path (unchanged when learned_sigma is off)
        samples_per_pixel = beam_on.sum() / image_size ** 2
        gain = samples_per_pixel * 2.0 * math.pi * sigma * sigma
        self.register_buffer("gain", torch.tensor(max(gain, 1e-3)))

        # P2: persistence decay parameter (only used when use_persistence)
        if self.use_persistence:
            init = cfg.persistence_init if cfg is not None else 0.1
            # Stored as logit so sigmoid keeps it in (0, 1)
            self.persistence_logit = nn.Parameter(
                torch.tensor(math.log(init / (1 - init)))
            )

    def forward(self, signal: torch.Tensor,
                sigma_per_step: Optional[torch.Tensor] = None,
                pos_deltas: Optional[torch.Tensor] = None) -> torch.Tensor:
        # signal:          (B, N, C) in [0, 1]
        # sigma_per_step:  (B, N)    in [sigma_min, sigma_max] (P1)
        # pos_deltas:      (B, N, 2) in [-perturb_max, perturb_max] (P3)
        b, n, c = signal.shape
        h = w = self.image_size

        sig = signal * self.beam_on_f.view(1, n, 1)

        # Beam positions: fixed path optionally perturbed (P3)
        if pos_deltas is not None:
            # pos_deltas: (B, N, 2) -> broadcasted with grid
            px = self.path_x.view(1, n, 1, 1) + pos_deltas[..., 0].view(b, n, 1, 1)
            py = self.path_y.view(1, n, 1, 1) + pos_deltas[..., 1].view(b, n, 1, 1)
            dx = self.grid_x.view(1, 1, h, w) - px
            dy = self.grid_y.view(1, 1, h, w) - py
        else:
            dx = self.grid_x.view(1, h, w) - self.path_x.view(n, 1, 1)
            dy = self.grid_y.view(1, h, w) - self.path_y.view(n, 1, 1)

        # Beam width: constant or per-step (P1)
        if sigma_per_step is not None:
            sig2 = (sigma_per_step ** 2).clamp(min=1e-4)  # (B, N)
            if pos_deltas is not None:
                # dx,dy already (B, N, H, W)
                sig2_b = sig2.view(b, n, 1, 1)
                weights = torch.exp(-(dx * dx + dy * dy) / (2.0 * sig2_b))
                # einsum across (B, N, H, W) and (B, N, C)
                img = torch.einsum("bnc,bnhw->bchw", sig, weights)
            else:
                # dx,dy are (N, H, W); broadcast sig2 across batch
                weights = torch.exp(
                    -(dx * dx + dy * dy).unsqueeze(0)
                    / (2.0 * sig2.view(b, n, 1, 1))
                )  # (B, N, H, W)
                img = torch.einsum("bnc,bnhw->bchw", sig, weights)
            # Per-step gain so signal magnitude maps to image magnitude
            # (approximate: integral of 2D gaussian = 2*pi*sigma^2 * samples_per_pixel)
            img = img / (sig2 * 2.0 * math.pi).mean()
        else:
            # V5 path: constant sigma
            if pos_deltas is not None:
                weights = torch.exp(-(dx * dx + dy * dy) / (2.0 * self.sigma ** 2))
                img = torch.einsum("bnc,bnhw->bchw", sig, weights) / self.gain
            else:
                weights = torch.exp(-(dx * dx + dy * dy) / (2.0 * self.sigma ** 2))
                img = torch.einsum("bnc,nhw->bchw", sig, weights) / self.gain

        # P2: phosphor persistence -- causal smoothing along scan order.
        # The rendered image is implicitly time-ordered by sample index,
        # but after einsum we already collapsed time. Persistence here is
        # applied as a 2D blur kernel proportional to the persistence weight,
        # which approximates the visual effect of phosphor afterglow.
        if self.use_persistence:
            alpha = torch.sigmoid(self.persistence_logit)
            # Light box blur (3x3 average) scaled by persistence
            blur = F.avg_pool2d(img, kernel_size=3, stride=1, padding=1)
            img = (1 - alpha) * img + alpha * blur

        return img.clamp(0.0, 1.0)


renderer = CRTRenderer(path_np, beam_on_np, cfg.image_size, cfg.beam_sigma, cfg).to(cfg.device)
print(f"renderer ready (persistence={renderer.use_persistence})")

renderer ready (persistence=False)


## 5. Image to signal conversion (unchanged from V5)

In [7]:
def image_to_signal(image: torch.Tensor, path_t: torch.Tensor,
                    beam_on_t: torch.Tensor) -> torch.Tensor:
    c, h, w = image.shape
    xs = 2.0 * path_t[:, 0] / (w - 1) - 1.0
    ys = 2.0 * path_t[:, 1] / (h - 1) - 1.0
    grid = torch.stack([xs, ys], dim=-1).view(1, -1, 1, 2)
    sampled = F.grid_sample(
        image.unsqueeze(0), grid, mode="bilinear", align_corners=True
    )
    signal = sampled.squeeze(-1).squeeze(0).T
    signal = signal.clamp(0.0, 1.0)
    signal = signal * beam_on_t.unsqueeze(-1)
    return signal

## 6. Dataset (unchanged from V5)

In [8]:
FLOWER_NAMES = [
    "pink primrose", "hard-leaved pocket orchid", "canterbury bells",
    "sweet pea", "english marigold", "tiger lily", "moon orchid",
    "bird of paradise", "monkshood", "globe thistle", "snapdragon",
    "colts foot", "king protea", "spear thistle", "yellow iris",
    "globe-flower", "purple coneflower", "peruvian lily", "balloon flower",
    "giant white arum lily", "fire lily", "pincushion flower", "fritillary",
    "red ginger", "grape hyacinth", "corn poppy", "prince of wales feathers",
    "stemless gentian", "artichoke", "sweet william", "carnation",
    "garden phlox", "love in the mist", "mexican aster", "alpine sea holly",
    "ruby-lipped cattleya", "cape flower", "great masterwort", "siam tulip",
    "lenten rose", "barbeton daisy", "daffodil", "sword lily", "poinsettia",
    "bolero deep blue", "wallflower", "marigold", "buttercup", "oxeye daisy",
    "common dandelion", "petunia", "wild pansy", "primula", "sunflower",
    "pelargonium", "bishop of llandaff", "gaura", "geranium", "orange dahlia",
    "pink-yellow dahlia", "cautleya spicata", "japanese anemone",
    "black-eyed susan", "silverbush", "californian poppy", "osteospermum",
    "spring crocus", "bearded iris", "windflower", "tree poppy", "gazania",
    "azalea", "water lily", "rose", "thorn apple", "morning glory",
    "passion flower", "lotus", "toad lily", "anthurium", "frangipani",
    "clematis", "hibiscus", "columbine", "desert-rose", "tree mallow",
    "magnolia", "cyclamen", "watercress", "canna lily", "hippeastrum",
    "bee balm", "ball moss", "foxglove", "bougainvillea", "camellia",
    "mallow", "mexican petunia", "bromelia", "blanket flower",
    "trumpet creeper", "blackberry lily",
]


class FlowerSignalDataset(Dataset):
    def __init__(self, hf_split, path_np, beam_on_np, image_size, label_names):
        from torchvision import transforms
        self.label_names = label_names
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
        ])

        path_t = torch.from_numpy(path_np)
        beam_on_t = torch.from_numpy(beam_on_np.astype(np.float32))

        self.signals = []
        self.captions = []
        for sample in hf_split:
            img = self.transform(sample["image"].convert("RGB"))
            sig = image_to_signal(img, path_t, beam_on_t)
            self.signals.append(sig.half())
            label = sample.get("label", 0)
            name = label_names[label] if label < len(label_names) else "flower"
            self.captions.append(f"a photo of a {name}")

    def __len__(self):
        return len(self.signals)

    def __getitem__(self, idx):
        return self.signals[idx].float(), self.captions[idx]

In [9]:
from datasets import load_dataset
hf = load_dataset("nelorth/oxford-flowers", split="train")
print(f"images: {len(hf)}")

dataset = FlowerSignalDataset(
    hf, path_np, beam_on_np, cfg.image_size, FLOWER_NAMES
)
print(f"cached signals: {len(dataset)}")

hf_test = load_dataset("nelorth/oxford-flowers", split="test")
test_dataset = FlowerSignalDataset(
    hf_test, path_np, beam_on_np, cfg.image_size, FLOWER_NAMES
)
print(f"test images: {len(test_dataset)}")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-12de94e121bdbe(…):   0%|          | 0.00/303M [00:00<?, ?B/s]

data/test-00000-of-00001-96eeec628415add(…):   0%|          | 0.00/43.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7169 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1020 [00:00<?, ? examples/s]

images: 7169
cached signals: 7169
test images: 1020


## 7. CLIP text encoder (unchanged from V5)

In [10]:
import clip


def load_clip(device):
    model, _ = clip.load("ViT-B/32", device=device)
    for p in model.parameters():
        p.requires_grad = False
    model.eval()
    return model


@torch.no_grad()
def encode_texts(clip_model, texts, device):
    tokens = clip.tokenize(texts, truncate=True).to(device)
    feats = clip_model.encode_text(tokens).float()
    return feats / feats.norm(dim=-1, keepdim=True)


clip_model = load_clip(cfg.device)

100%|████████████████████████████████████████| 338M/338M [00:01<00:00, 345MiB/s]


## 8. Positional encodings (unchanged from V5)

In [11]:
def sinusoidal_1d(n_positions: int, dim: int) -> torch.Tensor:
    pe = torch.zeros(n_positions, dim)
    position = torch.arange(n_positions, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(
        torch.arange(0, dim, 2, dtype=torch.float32) * -(math.log(10000.0) / dim)
    )
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


def sinusoidal_2d(coords: torch.Tensor, dim: int,
                  max_coord: float) -> torch.Tensor:
    half = dim // 2
    div_term = torch.exp(
        torch.arange(0, half, 2, dtype=torch.float32) * -(math.log(10000.0) / half)
    )
    x = coords[:, 0:1] / max_coord * math.pi * 2 * (max_coord / 2)
    y = coords[:, 1:2] / max_coord * math.pi * 2 * (max_coord / 2)

    pe = torch.zeros(coords.shape[0], dim)
    pe[:, 0:half:2] = torch.sin(x * div_term)
    pe[:, 1:half:2] = torch.cos(x * div_term)
    pe[:, half::2] = torch.sin(y * div_term)
    pe[:, half + 1::2] = torch.cos(y * div_term)
    return pe

## 9. FiLM conditioning (unchanged from V5)

In [12]:
class FiLM(nn.Module):
    def __init__(self, cond_dim: int, feature_dim: int):
        super().__init__()
        self.to_scale_shift = nn.Linear(cond_dim, feature_dim * 2)
        nn.init.zeros_(self.to_scale_shift.weight)
        nn.init.zeros_(self.to_scale_shift.bias)

    def forward(self, x: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        scale, shift = self.to_scale_shift(cond).chunk(2, dim=-1)
        return x * (1.0 + scale.unsqueeze(1)) + shift.unsqueeze(1)

## 10. Causal transformer block (unchanged from V5)

In [13]:
class CausalBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, ff_mult: int,
                 cond_dim: int, dropout: float, use_film: bool):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.use_film = use_film

        self.norm1 = nn.LayerNorm(d_model)
        self.qkv = nn.Linear(d_model, d_model * 3, bias=False)
        self.attn_out = nn.Linear(d_model, d_model)

        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * ff_mult),
            nn.GELU(),
            nn.Linear(d_model * ff_mult, d_model),
        )
        self.dropout = nn.Dropout(dropout)

        if use_film:
            self.film1 = FiLM(cond_dim, d_model)
            self.film2 = FiLM(cond_dim, d_model)

    def _attn(self, x: torch.Tensor) -> torch.Tensor:
        b, n, d = x.shape
        qkv = self.qkv(x).view(b, n, 3, self.n_heads, self.d_head)
        q, k, v = qkv.unbind(dim=2)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1, 2).contiguous().view(b, n, d)
        return self.attn_out(out)

    def forward(self, x: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        h = x + self.dropout(self._attn(self.norm1(x)))
        if self.use_film:
            h = self.film1(h, cond)
        h = h + self.dropout(self.ff(self.norm2(h)))
        if self.use_film:
            h = self.film2(h, cond)
        return h

## 11. Output heads

The DMoL head is unchanged from V5. V6 adds a separate auxiliary head that
predicts per-step `(sigma, dx, dy)` when the corresponding postulates are
enabled. The auxiliary head's outputs feed into the renderer; they do not
affect the DMoL likelihood.

Important design note: the auxiliary head outputs only contribute to the
*rendered* image, not to the AR likelihood directly. Likelihood is computed
on the DMoL output as in V5. This keeps bpd_pixel comparable across all
postulate configurations.

In [14]:
class DMoLHead(nn.Module):
    def __init__(self, d_model: int, channels: int, n_mix: int):
        super().__init__()
        self.channels = channels
        self.n_mix = n_mix
        self.out = nn.Linear(d_model, n_mix + channels * n_mix * 2)

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        return self.out(h)

    def _split(self, params: torch.Tensor):
        k = self.n_mix
        c = self.channels
        logit_probs = params[..., :k]
        rest = params[..., k:].view(*params.shape[:-1], c, k, 2)
        means = rest[..., 0]
        log_scales = rest[..., 1].clamp(min=-7.0)
        return logit_probs, means, log_scales

    def nll(self, params: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        logit_probs, means, log_scales = self._split(params)
        t = target.unsqueeze(-1)
        inv_s = torch.exp(-log_scales)
        centered = t - means
        bin_half = 0.5 / 255.0
        plus_in = inv_s * (centered + bin_half)
        min_in = inv_s * (centered - bin_half)
        cdf_plus = torch.sigmoid(plus_in)
        cdf_min = torch.sigmoid(min_in)
        log_cdf_plus = plus_in - F.softplus(plus_in)
        log_one_minus_cdf_min = -F.softplus(min_in)
        prob = cdf_plus - cdf_min
        log_prob_mid = torch.log(prob.clamp(min=1e-12))

        log_probs = torch.where(
            target.unsqueeze(-1) < 1e-3, log_cdf_plus,
            torch.where(
                target.unsqueeze(-1) > 1.0 - 1e-3,
                log_one_minus_cdf_min, log_prob_mid,
            ),
        )
        log_probs = log_probs.sum(dim=-2)
        log_mix = F.log_softmax(logit_probs, dim=-1)
        return -torch.logsumexp(log_mix + log_probs, dim=-1)

    @torch.no_grad()
    def sample(self, params: torch.Tensor, temperature: float = 1.0,
               top_p: float = 1.0) -> torch.Tensor:
        logit_probs, means, log_scales = self._split(params)

        if temperature <= 0:
            k_idx = logit_probs.argmax(dim=-1)
        else:
            scaled = logit_probs / max(temperature, 1e-6)
            probs = F.softmax(scaled, dim=-1)
            if top_p < 1.0:
                sorted_p, sorted_i = probs.sort(dim=-1, descending=True)
                cum = sorted_p.cumsum(dim=-1)
                mask = cum - sorted_p > top_p
                sorted_p = sorted_p.masked_fill(mask, 0.0)
                sorted_p = sorted_p / sorted_p.sum(dim=-1, keepdim=True)
                probs = torch.zeros_like(probs).scatter_(-1, sorted_i, sorted_p)
            k_idx = torch.distributions.Categorical(probs=probs).sample()

        k_idx_exp = k_idx.unsqueeze(-1).unsqueeze(-1).expand(
            *k_idx.shape, self.channels, 1
        )
        chosen_mean = means.gather(-1, k_idx_exp).squeeze(-1)
        chosen_log_s = log_scales.gather(-1, k_idx_exp).squeeze(-1)

        if temperature <= 0:
            sample = chosen_mean
        else:
            u = torch.rand_like(chosen_mean).clamp(1e-5, 1.0 - 1e-5)
            sample = chosen_mean + torch.exp(chosen_log_s) * (
                torch.log(u) - torch.log1p(-u)
            ) * temperature

        return sample.clamp(0.0, 1.0)


class BeamAuxHead(nn.Module):
    """Auxiliary head for V6 postulates.

    Produces per-step:
      sigma (P1, if enabled): mapped to [sigma_min, sigma_max] via sigmoid
      dx, dy (P3, if enabled): mapped to [-perturb_max, perturb_max] via tanh
    """

    def __init__(self, d_model: int, cfg: Config):
        super().__init__()
        self.cfg = cfg
        out_dim = 0
        if cfg.use_learned_sigma:
            out_dim += 1
        if cfg.use_beam_perturb:
            out_dim += 2
        self.enabled = out_dim > 0
        if self.enabled:
            self.proj = nn.Linear(d_model, out_dim)
            # Init to zero so initial outputs are at the centre of the range
            nn.init.zeros_(self.proj.weight)
            nn.init.zeros_(self.proj.bias)

    def forward(self, h: torch.Tensor):
        """Returns (sigma_per_step, pos_deltas), each None if disabled."""
        if not self.enabled:
            return None, None

        raw = self.proj(h)  # (B, N, out_dim)
        idx = 0
        sigma_per_step = None
        pos_deltas = None

        if self.cfg.use_learned_sigma:
            s_raw = raw[..., idx]
            sigma_per_step = (
                self.cfg.sigma_min
                + (self.cfg.sigma_max - self.cfg.sigma_min) * torch.sigmoid(s_raw)
            )
            idx += 1

        if self.cfg.use_beam_perturb:
            d_raw = raw[..., idx:idx + 2]
            pos_deltas = self.cfg.beam_perturb_max * torch.tanh(d_raw)
            idx += 2

        return sigma_per_step, pos_deltas

## 12. Full model

The transformer backbone is unchanged from V5. The forward pass returns DMoL
parameters as before; if any V6 postulate is enabled, it also returns the
auxiliary outputs (sigma_per_step, pos_deltas) which the caller passes to
the renderer when they want a rendered image.

Two important points:

1. The DMoL likelihood is computed on the same target signals as V5, so
   `bpd_pixel` numbers are directly comparable across all V6 ablations.
2. The auxiliary head's outputs only matter when the model renders an image
   (generation, teacher-forced reconstruction). They do not influence the
   training NLL.

In [15]:
class SignalTransformer(nn.Module):
    def __init__(self, cfg: Config, path: np.ndarray, beam_on: np.ndarray):
        super().__init__()
        self.cfg = cfg
        seq_len = len(path)
        input_ch = cfg.channels + (1 if cfg.use_flyback_mask else 0)

        self.input_proj = nn.Linear(input_ch, cfg.d_model)
        self.cond_proj = nn.Sequential(
            nn.Linear(cfg.clip_dim, cfg.cond_dim),
            nn.GELU(),
            nn.Linear(cfg.cond_dim, cfg.cond_dim),
        )

        seq_pe = sinusoidal_1d(seq_len, cfg.d_model)
        self.register_buffer("seq_pe", seq_pe)

        path_t = torch.from_numpy(path)
        path_pe = sinusoidal_2d(path_t, cfg.d_model, max_coord=cfg.image_size - 1)
        self.register_buffer("path_pe", path_pe)

        self.register_buffer("beam_on_f",
                             torch.from_numpy(beam_on.astype(np.float32)))

        self.blocks = nn.ModuleList([
            CausalBlock(cfg.d_model, cfg.n_heads, cfg.ff_mult,
                        cfg.cond_dim, cfg.dropout, cfg.use_film)
            for _ in range(cfg.n_layers)
        ])
        self.norm_out = nn.LayerNorm(cfg.d_model)
        self.head = DMoLHead(cfg.d_model, cfg.channels, cfg.n_mixtures)
        self.aux_head = BeamAuxHead(cfg.d_model, cfg)

    def _prepare_input(self, signal_prev: torch.Tensor) -> torch.Tensor:
        if self.cfg.use_flyback_mask:
            mask = self.beam_on_f[: signal_prev.size(1)]
            mask = mask.view(1, -1, 1).expand(signal_prev.size(0), -1, 1)
            return torch.cat([signal_prev, mask], dim=-1)
        return signal_prev

    def _trunk(self, shifted_signal: torch.Tensor,
               clip_emb: torch.Tensor) -> torch.Tensor:
        x = self._prepare_input(shifted_signal)
        b, n, _ = x.shape
        h = self.input_proj(x)

        if self.cfg.use_seq_pos_enc:
            h = h + self.seq_pe[:n].unsqueeze(0)
        if self.cfg.use_path_pos_enc:
            h = h + self.path_pe[:n].unsqueeze(0)

        cond = self.cond_proj(clip_emb)
        for block in self.blocks:
            h = block(h, cond)
        return self.norm_out(h)

    def forward(self, shifted_signal: torch.Tensor,
                clip_emb: torch.Tensor):
        """Returns (dmol_params, sigma_per_step, pos_deltas).

        sigma_per_step and pos_deltas are None if the corresponding postulate
        is disabled.
        """
        h = self._trunk(shifted_signal, clip_emb)
        params = self.head(h)
        sigma_per_step, pos_deltas = self.aux_head(h)
        return params, sigma_per_step, pos_deltas

    @torch.no_grad()
    def generate(self, clip_emb: torch.Tensor, n_steps: int,
                 temperature: float = 1.0, top_p: float = 1.0):
        """Returns (signals, sigma_seq, pos_seq) -- the trailing two are
        None if the corresponding postulate is disabled.
        """
        device = clip_emb.device
        b = clip_emb.size(0)
        c = self.cfg.channels

        cond = self.cond_proj(clip_emb)
        generated = torch.zeros(b, 0, c, device=device)
        sigma_acc = [] if self.cfg.use_learned_sigma else None
        pos_acc = [] if self.cfg.use_beam_perturb else None
        prev = torch.zeros(b, 1, c, device=device)

        seq_pe = self.seq_pe.unsqueeze(0)
        path_pe = self.path_pe.unsqueeze(0)

        for i in range(n_steps):
            inp = torch.cat([prev, generated], dim=1) if generated.size(1) else prev
            x = self._prepare_input(inp)
            h = self.input_proj(x)
            if self.cfg.use_seq_pos_enc:
                h = h + seq_pe[:, : h.size(1)]
            if self.cfg.use_path_pos_enc:
                h = h + path_pe[:, : h.size(1)]
            for block in self.blocks:
                h = block(h, cond)
            h = self.norm_out(h)
            last = h[:, -1:, :]
            params = self.head(last)
            s_step, d_step = self.aux_head(last)

            next_sample = self.head.sample(params, temperature=temperature,
                                           top_p=top_p)
            if self.cfg.use_flyback_mask and not bool(self.beam_on_f[i]):
                next_sample = torch.zeros_like(next_sample)
            generated = torch.cat([generated, next_sample], dim=1)
            if sigma_acc is not None:
                sigma_acc.append(s_step.squeeze(1))  # (B,)
            if pos_acc is not None:
                pos_acc.append(d_step.squeeze(1))    # (B, 2)

        sigma_seq = torch.stack(sigma_acc, dim=1) if sigma_acc else None
        pos_seq = torch.stack(pos_acc, dim=1) if pos_acc else None
        return generated, sigma_seq, pos_seq

## 13. Training

V6 training is unchanged from V5 in structure, with one addition: when
`use_beam_perturb` is True, we add a small L2 penalty on the position
deltas so the model does not drift far from the canonical raster path.
This keeps the path PE meaningful even with perturbation enabled.

In [16]:
LN2 = math.log(2.0)


@torch.no_grad()
def evaluate(model, clip_model, dataset, n_batches=20):
    model.eval()
    loader = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=False,
                        num_workers=0, drop_last=True)
    beam_on_f = model.beam_on_f

    total_nll_nat = 0.0
    total_images = 0
    total_mse = 0.0
    total_active_samples = 0

    image_dims = cfg.image_size * cfg.image_size * cfg.channels

    for i, (signals, captions) in enumerate(loader):
        if i >= n_batches:
            break
        signals = signals.to(cfg.device)
        b, n, c = signals.shape
        clip_emb = encode_texts(clip_model, list(captions), cfg.device)
        shifted = torch.cat(
            [torch.zeros(b, 1, c, device=cfg.device), signals[:, :-1, :]], dim=1,
        )
        params, _, _ = model(shifted, clip_emb)
        nll = model.head.nll(params, signals)
        mask = beam_on_f.view(1, -1).expand(b, -1)

        nll_per_image = (nll * mask).sum(dim=1)
        total_nll_nat += nll_per_image.sum().item()
        total_images += b

        pred = model.head.sample(params, temperature=0.0)
        mse = ((pred - signals) ** 2 * mask.unsqueeze(-1)).sum().item()
        total_mse += mse
        total_active_samples += mask.sum().item() * c

    mean_nll_per_image = total_nll_nat / max(total_images, 1)
    bpd_pixel = mean_nll_per_image / LN2 / image_dims
    bpd_signal = total_nll_nat / max(total_active_samples * LN2, 1e-9)
    mse_per = total_mse / max(total_active_samples, 1)
    psnr = 10.0 * math.log10(1.0 / max(mse_per, 1e-12))

    return {
        "nll_per_image_nats": mean_nll_per_image,
        "bpd_pixel": bpd_pixel,
        "bpd_signal": bpd_signal,
        "psnr_db": psnr,
    }

In [17]:
CKPT_DIR = f"{WORK_DIR}/checkpoints_v6"
os.makedirs(CKPT_DIR, exist_ok=True)


# Separate dirs so Section 15 does not clash with ablations / resume mistakes
BASELINE_CKPT_DIR = f"{WORK_DIR}/checkpoints_v6_baseline_30ep"
ABLATION_CKPT_ROOT = f"{WORK_DIR}/checkpoints_v6_ablation"


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def save_checkpoint(model, optim, epoch, step, path):
    torch.save({
        "model": model.state_dict(),
        "optim": optim.state_dict(),
        "epoch": epoch,
        "step": step,
    }, path)


def load_checkpoint(model, optim, path, device):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model"])
    optim.load_state_dict(ckpt["optim"])
    return ckpt["epoch"], ckpt["step"]


def cosine_warmup_lr(step, warmup, total, base_lr):
    if step < warmup:
        return base_lr * step / max(warmup, 1)
    progress = (step - warmup) / max(total - warmup, 1)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


def train(cfg: Config, model: SignalTransformer, clip_model,
          dataset: Dataset, val_dataset: Dataset = None,
          resume_from: str = None,
          perturb_reg: float = 1e-3,
          checkpoint_dir: str = None) -> None:
    """V6 training adds an optional L2 regularizer on predicted position
    deltas to keep beam_perturb close to the canonical path.
    """
    from tqdm.auto import tqdm

    ckpt_root = checkpoint_dir if checkpoint_dir else CKPT_DIR
    os.makedirs(ckpt_root, exist_ok=True)

    loader = DataLoader(
        dataset, batch_size=cfg.batch_size, shuffle=True,
        num_workers=0, drop_last=True, pin_memory=True,
    )
    optim = torch.optim.AdamW(
        model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay,
        betas=(0.9, 0.95),
    )
    total_steps = len(loader) * cfg.epochs
    beam_on_f = model.beam_on_f
    scaler = torch.amp.GradScaler(cfg.device) if cfg.device == "cuda" else None

    start_epoch, step = 1, 0
    best_val = float("inf")
    if resume_from and os.path.exists(resume_from):
        start_epoch, step = load_checkpoint(model, optim, resume_from, cfg.device)
        start_epoch += 1
        print(f"resumed from epoch {start_epoch - 1}, step {step}")

    for epoch in range(start_epoch, cfg.epochs + 1):
        model.train()
        running, running_n = 0.0, 0
        pbar = tqdm(loader, desc=f"epoch {epoch}/{cfg.epochs}")
        for signals, captions in pbar:
            signals = signals.to(cfg.device, non_blocking=True)
            b, n, c = signals.shape

            with torch.no_grad():
                clip_emb = encode_texts(clip_model, list(captions), cfg.device)

            p_ss = cfg.ss_max * min(step / max(total_steps, 1), 1.0)
            shifted = torch.cat(
                [torch.zeros(b, 1, c, device=cfg.device), signals[:, :-1, :]], dim=1,
            )
            if p_ss > 0:
                with torch.no_grad():
                    preview_params, _, _ = model(shifted, clip_emb)
                    preview = model.head.sample(preview_params, temperature=1.0)
                    mask = (torch.rand(b, n, 1, device=cfg.device) < p_ss).float()
                    mixed = shifted.clone()
                    mixed[:, 1:, :] = (
                        mask[:, 1:, :] * preview[:, :-1, :]
                        + (1.0 - mask[:, 1:, :]) * shifted[:, 1:, :]
                    )
                    shifted = mixed

            lr = cosine_warmup_lr(step, cfg.warmup_steps, total_steps, cfg.lr)
            for g in optim.param_groups:
                g["lr"] = lr

            optim.zero_grad(set_to_none=True)
            if scaler is not None:
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    params, _, pos_deltas = model(shifted, clip_emb)
                    nll = model.head.nll(params, signals)
                    mask_b = beam_on_f.view(1, -1).expand(b, -1)
                    loss = (nll * mask_b).sum() / mask_b.sum().clamp(min=1.0)
                    if pos_deltas is not None and perturb_reg > 0:
                        loss = loss + perturb_reg * (pos_deltas ** 2).mean()
                scaler.scale(loss).backward()
                scaler.unscale_(optim)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                scaler.step(optim)
                scaler.update()
            else:
                params, _, pos_deltas = model(shifted, clip_emb)
                nll = model.head.nll(params, signals)
                mask_b = beam_on_f.view(1, -1).expand(b, -1)
                loss = (nll * mask_b).sum() / mask_b.sum().clamp(min=1.0)
                if pos_deltas is not None and perturb_reg > 0:
                    loss = loss + perturb_reg * (pos_deltas ** 2).mean()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                optim.step()

            running += loss.item() * b
            running_n += b
            step += 1
            pbar.set_postfix(nll=f"{loss.item():.4f}", lr=f"{lr:.2e}", ss=f"{p_ss:.2f}")

        avg = running / running_n
        print(f"epoch {epoch}/{cfg.epochs} avg_nll={avg:.4f}")
        save_checkpoint(model, optim, epoch, step, f"{ckpt_root}/latest.pt")
        if epoch % 5 == 0:
            save_checkpoint(model, optim, epoch, step, f"{ckpt_root}/epoch_{epoch}.pt")

        if val_dataset is not None and epoch % 5 == 0:
            val_m = evaluate(model, clip_model, val_dataset, n_batches=10)
            if val_m["nll_per_image_nats"] < best_val:
                best_val = val_m["nll_per_image_nats"]
                save_checkpoint(model, optim, epoch, step, f"{ckpt_root}/best.pt")
            print(f"  val nll={val_m['nll_per_image_nats']:.4f}  bpd_pixel={val_m['bpd_pixel']:.3f}"
                  f"  psnr={val_m['psnr_db']:.2f}  best={best_val:.4f}")


## 14. Sanity check: V6 with all postulates off == V5

Before running any postulate ablations, verify that the V6 code with all
toggles set to False reproduces the V5 baseline behaviour. Train for 5 epochs
and check that bpd_pixel matches V5's epoch-5 number on the same seed.

If this matches, we know any difference observed when we flip a postulate
toggle is due to the postulate, not a regression in V6.

In [18]:
# Quick sanity: confirm postulate-free V6 has same param count as V5.
sanity_model = SignalTransformer(cfg, path_np, beam_on_np).to(cfg.device)
n_params = sum(p.numel() for p in sanity_model.parameters() if p.requires_grad)
print(f"V6 (no postulates) params: {n_params:,}")
print(f"Aux head enabled: {sanity_model.aux_head.enabled}  "
      f"(should be False with no postulates)")
del sanity_model

V6 (no postulates) params: 6,520,867
Aux head enabled: False  (should be False with no postulates)


## 15. Build and train (V5 best config = `no_seq_pe`)

This run reproduces V5's headline finding: train with `use_seq_pos_enc=False`
and confirm bpd_pixel matches the V5 V4 result (~7.06 at 30 epochs).

In [19]:
import copy
best_cfg = copy.deepcopy(cfg)
best_cfg.use_seq_pos_enc = False
best_cfg.epochs = 30

torch.manual_seed(0)
np.random.seed(0)
model = SignalTransformer(best_cfg, path_np, beam_on_np).to(best_cfg.device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"parameters: {n_params:,}")

ensure_dir(BASELINE_CKPT_DIR)
train(best_cfg, model, clip_model, dataset,
      val_dataset=test_dataset,
      resume_from=f"{BASELINE_CKPT_DIR}/latest.pt",
      checkpoint_dir=BASELINE_CKPT_DIR)

print("\n=== V6 baseline (no_seq_pe) test ===")
print(evaluate(model, clip_model, test_dataset, n_batches=20))


parameters: 6,520,867


epoch 1/30:   0%|          | 0/298 [00:00<?, ?it/s]

epoch 1/30 avg_nll=14.9354


epoch 2/30:   0%|          | 0/298 [00:00<?, ?it/s]

epoch 2/30 avg_nll=13.0140


...

epoch 28/30 avg_nll=7.9447


epoch 29/30:   0%|          | 0/298 [00:00<?, ?it/s]

epoch 29/30 avg_nll=7.9785


epoch 30/30:   0%|          | 0/298 [00:00<?, ?it/s]

epoch 30/30 avg_nll=8.0194
  val nll=13576.6852  bpd_pixel=6.376  psnr=32.30  best=13576.6852

=== V6 baseline (no_seq_pe) test ===
{'nll_per_image_nats': 13451.136393229166, 'bpd_pixel': 6.317020758083586, 'bpd_signal': 3.158510379041793, 'psnr_db': 32.42296876618136}


## 16. Postulate ablations (P1, P2, P3)

Each postulate is tested independently against the V5 best config
(`use_seq_pos_enc=False`, all V6 toggles off). For each postulate:

1. Train with `seeds=(0,1,2)` for 10 epochs.
2. Report mean and std of NLL per image.
3. Compute sigma against the matched V5 baseline (also at 10 epochs, 3 seeds).
4. Decision: keep if delta_nll < -3*sigma_pooled.

Total runtime estimate: 4 configs (baseline + P1 + P2 + P3) x 3 seeds x 10
epochs = ~6 hours on T4.

**Note on P2 (persistence):** the DMoL loss is computed on **scan signals**, not on the differentiable renderer output. A persistence layer in `CRTRenderer` mainly affects **rendered** images unless you add an extra image-space loss — so do not expect large `bpd_pixel` shifts from P2 in this setup alone.

**Checkpoints:** each ablation seed writes to `checkpoints_v6_ablation/<run_name>/seed_<s>/` so runs do not overwrite each other.



In [20]:
import copy
import statistics


def run_one(name: str, override: dict, epochs: int = 10, seed: int = 0,
            base_cfg: Optional[Config] = None):
    if base_cfg is None:
        base_cfg = cfg
    abl_cfg = copy.deepcopy(base_cfg)
    abl_cfg.use_seq_pos_enc = False  # Always start from V5 best
    abl_cfg.epochs = epochs
    for k, v in override.items():
        setattr(abl_cfg, k, v)

    torch.manual_seed(seed)
    np.random.seed(seed)
    abl_model = SignalTransformer(abl_cfg, path_np, beam_on_np).to(abl_cfg.device)
    run_ckpt = f"{ABLATION_CKPT_ROOT}/{name}/seed_{seed}"
    ensure_dir(run_ckpt)
    train(abl_cfg, abl_model, clip_model, dataset,
          val_dataset=test_dataset,
          checkpoint_dir=run_ckpt)
    metrics = evaluate(abl_model, clip_model, test_dataset, n_batches=20)
    n_params = sum(p.numel() for p in abl_model.parameters() if p.requires_grad)
    metrics["n_params"] = n_params
    return abl_model, metrics


def run_with_seeds(name: str, override: dict, seeds=(0, 1, 2), epochs: int = 10):
    nlls, bpds, psnrs = [], [], []
    for s in seeds:
        _, m = run_one(name, override, epochs=epochs, seed=s)
        nlls.append(m["nll_per_image_nats"])
        bpds.append(m["bpd_pixel"])
        psnrs.append(m["psnr_db"])
    return {
        "nll_mean": statistics.mean(nlls),
        "nll_std": statistics.stdev(nlls) if len(nlls) > 1 else 0.0,
        "bpd_mean": statistics.mean(bpds),
        "bpd_std": statistics.stdev(bpds) if len(bpds) > 1 else 0.0,
        "psnr_mean": statistics.mean(psnrs),
        "psnr_std": statistics.stdev(psnrs) if len(psnrs) > 1 else 0.0,
    }


# ----- Postulate ablation grid (uncomment to run; ~6 hours on T4) -----
postulate_grid = [
    ("baseline_no_seq_pe", {}),
    ("P1_learned_sigma",   {"use_learned_sigma": True}),
    ("P2_persistence",     {"use_persistence": True}),
    ("P3_beam_perturb",    {"use_beam_perturb": True}),
]

results = {}
for name, override in postulate_grid:
    print(f"\n========== {name} ==========")
    results[name] = run_with_seeds(name, override, seeds=(0, 1, 2), epochs=10)
    r = results[name]
    print(f"  nll  = {r['nll_mean']:.1f} +/- {r['nll_std']:.1f}")
    print(f"  bpd  = {r['bpd_mean']:.4f} +/- {r['bpd_std']:.4f}")
    print(f"  psnr = {r['psnr_mean']:.2f} +/- {r['psnr_std']:.2f}")



========== baseline_no_seq_pe ==========


epoch 1/10:   0%|          | 0/298 [00:00<?, ?it/s]

epoch 1/10 avg_nll=14.9264


epoch 2/10:   0%|          | 0/298 [00:00<?, ?it/s]

epoch 2/10 avg_nll=13.0086


...

epoch 8/10 avg_nll=9.8082


epoch 9/10:   0%|          | 0/298 [00:00<?, ?it/s]

epoch 9/10 avg_nll=9.5948


epoch 10/10:   0%|          | 0/298 [00:00<?, ?it/s]

epoch 10/10 avg_nll=9.5667
  val nll=17906.4327  bpd_pixel=8.409  psnr=28.84  best=17906.4327
  nll  = 17499.1 +/- 266.7
  bpd  = 8.2180 +/- 0.1252
  psnr = 29.41 +/- 0.54


## 17. Decision table

For each postulate, compute `(nll_mean - baseline_nll) / pooled_std`. A negative
value means the postulate helps; a value with magnitude > 3 is the threshold
for adoption.

In [21]:
def decide(results, baseline_key="baseline_no_seq_pe", threshold=3.0):
    base = results[baseline_key]
    print(f"{'postulate':24s}  {'delta_nll':>12s}  {'pooled_std':>10s}  "
          f"{'sigma':>8s}  {'verdict':>10s}")
    for name, r in results.items():
        if name == baseline_key:
            continue
        delta = r["nll_mean"] - base["nll_mean"]
        pooled = math.sqrt(0.5 * (r["nll_std"] ** 2 + base["nll_std"] ** 2))
        sigma = delta / pooled if pooled > 0 else 0.0
        if sigma < -threshold:
            verdict = "ADOPT"
        elif sigma > threshold:
            verdict = "HARMFUL"
        else:
            verdict = "no effect"
        print(f"{name:24s}  {delta:+12.2f}  {pooled:10.2f}  "
              f"{sigma:+8.2f}  {verdict:>10s}")


decide(results)

postulate                    delta_nll  pooled_std     sigma     verdict
P1_learned_sigma               -390.52      601.22     -0.65   no effect
P2_persistence                  +90.95      871.08     +0.10   no effect
P3_beam_perturb                -363.83      616.28     -0.59   no effect


## 18. What V6 establishes (and doesn't)

**Established by V6:**
- A multi-seed ablation protocol for three CRT-inspired architectural
  choices (P1 learned sigma, P2 persistence, P3 beam perturbation).
- Each postulate is tested against the V5 best config (`no_seq_pe`) at
  matched seeds and epochs.
- Postulates are adopted only if they improve NLL by more than 3 pooled
  standard deviations.

**Not yet established by V6:**
- Comparison against matched-parameter PixelCNN++ on Flowers-102 is available
  in `pixelcnn_baseline_flowers.ipynb` (run separately); cite those numbers as
  the pixel-space reference.
- Generalization to CIFAR-10. Same architecture, different distribution.
  Required if we want comparison to published PixelCNN++ numbers.
- Scaling beyond 32x32. Path PE finding tested at 48x48 in V5 already;
  if any V6 postulate is adopted, retest at 48x48 too.

**Negative results from V6 must be reported.** If P1/P2/P3 fail to improve
bpd_pixel, that is publication-worthy in itself: it tells the community
that obvious CRT-inspired modifications do not help on raster-scan AR with
DMoL likelihood.

**Path forward (in order):**
1. Run sections 15-17 to confirm V5 baseline reproduces and run postulate grid.
2. For any postulate that adopts, retrain at 30 epochs to update headline.
3. Implement matched-param PixelCNN++ on same Flowers split.
4. Optional: CIFAR-10 transfer.
5. Write paper with clear table: V5 baseline, V6 best, PixelCNN++ baseline.

**The co-author's parallel notebooks** (Conv1D-VAE, physics-VAE) explore
the VAE direction but are excluded from this paper because they use a
different objective (L1 + KL, no tractable likelihood) and cannot produce
bits-per-dim numbers comparable to PixelCNN++. They may serve as appendix
material for qualitative samples, not headline numbers.